# ollada — steps sweep for diffcoder:8b on a T4 (notebook v0.1.0)

**Before you start:** Runtime → Change runtime type → **T4 GPU** (High-RAM if offered), then Runtime → Run all.

The verified default is steps=128 (the vendor example uses 512) and it produced 6.7 tok/s on the T4 — steps are the speed knob. Steps must divide the 64 blocks (gen_length 256 / block_length 4), hence 64/128/256. 512 is excluded: at T4 speeds it would take ~3 min per prompt and teach us nothing 256 doesn't.

Same fixed prompts as the verification run, several step counts, speed recorded per
config. **Decision rule:** the new recipe default is the lowest step count whose
outputs show no visible quality loss against the highest count. Send Muse the
SWEEP RECORD (last cell) and the printed outputs. Expected: ~8 min of generation after the load.

In [ ]:
# 1 — install (DiffCoder family: transformers 5.3.0 (the version its config was saved with))
import subprocess, sys
pkgs = [
    "transformers==5.3.0", "torch", "accelerate", "bitsandbytes", "pyyaml",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "git+https://github.com/OpenSourceRulesz/ollada@main"], check=True)
print("installed")

In [ ]:
# 2 — version stamp + GPU check (acceptance gate)
import torch, ollada
NOTEBOOK_VERSION = "0.1.0"  # also used by the record cell — keep it a variable, not a literal
print("notebook version:", NOTEBOOK_VERSION)
print("ollada version:", ollada.__version__)
assert torch.cuda.is_available(), "no CUDA — set the runtime to T4 GPU first"
name = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name}  VRAM: {total:.1f} GB")

In [ ]:
# 3 — load diffcoder:8b through ollada itself (timed)
import time, torch
from ollada.runtime.manager import ModelManager

# Rerun hygiene: drop stale model references from a previous run in this
# kernel before loading (a second resident copy OOMs a 16GB card).
import gc
for _stale in ("adapter", "manager"):
    globals().pop(_stale, None)
gc.collect()
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
manager = ModelManager()
adapter = manager.ensure_loaded("diffcoder:8b")
load_s = time.time() - t0
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"loaded diffcoder:8b in {load_s:.1f}s — peak VRAM {peak:.2f} GB")
print("ready on cuda")

In [ ]:
# 4 — the sweep: same prompts, each step count (current default: 128)
prompts = [
    "Write a Python function that returns the nth Fibonacci number, with a docstring.",
    "This function is supposed to return the average of a list of numbers, but it crashes on an empty list and has another bug. Find and fix both bugs, and show the corrected function:\n\ndef average(nums):\n    total = 0\n    for n in nums:\n        total =+ n\n    return total / len(nums)",
    "Complete this Python function so it checks whether a string is a palindrome, ignoring case and non-alphanumeric characters. Only output the completed code:\n\ndef is_palindrome(s):\n    # your code here",
    "In one or two sentences, what does a hash map do?",
]
STEPS = [64, 128, 256]
rows = []
for steps in STEPS:
    for p in prompts:
        r = manager.chat("diffcoder:8b", [{"role": "user", "content": p}], overrides={"steps": steps})
        tps = r.tokens / r.seconds if r.seconds else 0
        rows.append({"steps": steps, "prompt": p[:40], "tokens": r.tokens,
                      "seconds": round(r.seconds, 2), "tok_per_s": round(tps, 1)})
        print("=" * 70)
        print(f"STEPS={steps}  PROMPT: {p[:60]}")
        print(f"[{r.tokens} tokens in {r.seconds:.1f}s = {tps:.1f} tok/s]")
        print(r.text)
print("SWEEP DONE")

In [ ]:
# 5 — SWEEP RECORD (paste this whole block back to Muse)
import json, torch
import ollada as _ollada
record = {
    "kind": "steps_sweep",
    "model": "diffcoder:8b",
    "notebook_version": NOTEBOOK_VERSION,
    "ollada_version": _ollada.__version__,
    "hardware": torch.cuda.get_device_name(0),
    "load_seconds": round(load_s, 1),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "rows": rows,
}
print(json.dumps(record, indent=2))